# 04. A Simple Dynamics Loop In Pinocchio

This notebook keeps the dynamics story intentionally small: simulate one robot with forward dynamics, compare `pin.aba(...)` against an explicit solve of `M(q) a = 	au - h(q, v)`, and then add a simple damping term.

## What you should remember
- `pin.aba(...)` gives forward dynamics efficiently.
- `pin.crba(...)` gives the mass matrix and `pin.nle(...)` gives the nonlinear effects term.
- In the no-contact case, a hand-written solve of `M(q) a = 	au - h(q, v)` should match `pin.aba(...)` very closely.
- `pin.integrate(...)` updates the configuration safely after each velocity update.

In [1]:
# %%capture
# !pip install pin viser robot_descriptions numpy scipy matplotlib trimesh

import site
site.main()

In [2]:
# %%capture
# !wget -O viz.py https://raw.githubusercontent.com/Atarilab/colab_utils/refs/heads/main/viz.py
import viz

If the next cell gives an error restart the session to load the libraries (ctrl + m + .) or click runtime -> restart session. Then rerun the second and third codeblocks (do not rerun the first block!).

In [3]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pinocchio as pin
from robot_descriptions.loaders.pinocchio import load_robot_description

from viz import PinNotebookViz, create_server

server, share_url = create_server()
notebook_viz = PinNotebookViz(server)

print(f"Open the visualizer here: {share_url}")

╭────── viser (listening *:8083) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8083   │
│   Websocket │ ws://localhost:8083     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://keyframe-reshape.share.viser.studio

Open the visualizer here: https://keyframe-reshape.share.viser.studio


(viser) Connection opened (0, 1 total), 6 persistent messages

In [5]:
robot = load_robot_description("ur5_description")
model = robot.model
data = model.createData()

notebook_viz.attach_robot(robot)
q0 = np.array([0.0, -1.2, 1.2, -0.8, 0.0, 0.0])
v0 = np.zeros(model.nv)
notebook_viz.display(q0)

print(f"nq = {model.nq}, nv = {model.nv}")

nq = 6, nv = 6


## The equation we will solve
Ignoring contact forces, the rigid-body dynamics equation is

$$
M(q) a = 	\tau - h(q, v).
$$

Here `h(q, v)` is the nonlinear effects vector. In practice it collects the terms coming from gravity, Coriolis effects, and centrifugal effects.

In [6]:
M = pin.crba(model, data, q0)
h = pin.nle(model, data, q0, v0)

print("Mass matrix shape:", M.shape)
print("First row of M:")
print(M[0])
print()
print("Nonlinear effects vector h(q, v):")
print(h)

Mass matrix shape: (6, 6)
First row of M:
[ 1.88032275e+00 -3.79384780e-01  1.35244973e-03  1.35244973e-03
 -1.76435400e-01  0.00000000e+00]

Nonlinear effects vector h(q, v):
[-1.26441080e-15 -3.15668250e+01 -1.58089843e+01 -1.25155862e-01
  0.00000000e+00  0.00000000e+00]


## Forward dynamics with `pin.aba`
Pinocchio can solve the forward dynamics directly:

$$
a = \mathrm{aba}(\text{model}, \text{data}, q, v).
$$

We will use a small sinusoidal torque on one joint and integrate the resulting motion.

In [7]:
def simulate(acceleration_function, steps=1000, dt=0.01):
    q = q0.copy()
    v = v0.copy()
    trajectory = []
    for k in range(steps):
        tau = np.zeros(model.nv)
        a = acceleration_function(q, v, tau)
        v = v + a * dt
        q = pin.integrate(model, q, v * dt)
        trajectory.append(q.copy())
    return trajectory

acceleration_function = lambda q, v, tau: pin.aba(model, data, q, v, tau)

trajectory_aba = simulate(acceleration_function)

for q_k in trajectory_aba[:]:
    notebook_viz.display(q_k)
    time.sleep(0.01)

## Exercise
Given the rigid body dynamics function:
$$
M(q) a = 	\tau - h(q, v).
$$
solve for acceleration using pinocchio's $M(q)$ and $h(q, v)$ and compare with pin.aba.

In [ ]:
def forward_dynamics_explicit(todo):
    todo
    return 


q_test = np.array([0.3, -1.0, 1.1, -0.5, 0.2, -0.1])
v_test = np.array([0.0, 0.4, -0.2, 0.1, -0.1, 0.3])
tau_test = np.array([0.2, -0.8, 0.5, 0.1, -0.2, 0.3])

a_aba = pin.aba(model, data, q_test, v_test, tau_test)
a_explicit = forward_dynamics_explicit(todo)

print("a from ABA:")
print(a_aba)
print()
print("a from explicit solve:")
print(a_explicit)
print()
print("Difference norm:", np.linalg.norm(a_aba - a_explicit))

## Exercise
Take the quadruped with the attached arm from notebook 1 and build a similar simulation loop for the combined system. Try both a fixed-base version and a free-flyer version of the quadruped.